# Validation: CPU (SciPy) vs JAX GPU Solver

This notebook runs the NEE solver with both the original SciPy CPU backend and the
JAX JIT-compiled GPU backend, comparing results side-by-side.

We walk up a **validation ladder** of increasing physical complexity,
verifying that the two solvers agree at each level.

In [ ]:
import numpy as np
from numpy.fft import fft, ifft, fftshift, fftfreq
import matplotlib.pyplot as plt
import time

import snow.pulses as pulses
import snow.materials as materials
import snow.waveguides as waveguides

from scipy.constants import pi, c
nm = 1e-9
um = 1e-6
mm = 1e-3
ps = 1e-12
fs = 1e-15
MHz = 1e6
THz = 1e12
pJ = 1e-12
uW = 1e-6

plt.rcParams.update({'font.size': 14})

## Setup

Same grid and waveguide as Tutorial 5 (SHG).

In [ ]:
# Time/Frequency grid
lam_start = 800*nm
lam_stop = 3*um
f_max = c/lam_start
f_min = c/lam_stop
BW = f_max - f_min
N = 2**10
dt = 1/BW
T = N/BW
t = -T/2 + np.arange(0, T, step=dt)
f = fftfreq(N, dt)
f_ref = (f_max + f_min)/2
f_abs = f + f_ref
wl = c/f_abs

# Waveguide
pp = 5.18*um

def make_waveguide(L, X0, alpha=0, poling_fn=None):
    wg = waveguides.waveguide(w_top=1800*nm, h_thinfilm=700*nm, h_etch=350*nm,
                              tf_material='LN_MgO_e', box_material='SiO2', clad_material='Air')
    if poling_fn is None:
        poling_fn = lambda z: np.sign(np.cos(z*2*pi/pp))
    wg.add_poling(poling_fn)
    wg.set_nonlinear_coeffs(N=1, X0=X0)
    wg.set_length(L)
    wg.set_loss(alpha)
    return wg

v_ref = 1/waveguides.waveguide(w_top=1800*nm, h_thinfilm=700*nm, h_etch=350*nm).beta1(1*um)
print(f'Grid: N={N}, BW={BW/THz:.1f} THz, dt={dt/fs:.2f} fs')

## Helper: Run both backends and compare

In [ ]:
def run_and_compare(title, pump, wg, v_ref=v_ref):
    """Run CPU and JAX, plot side-by-side using pulse convenience methods."""
    
    # CPU (SciPy)
    t0 = time.perf_counter()
    out_cpu, steps_cpu = wg.propagate_NEE(pump, v_ref=v_ref, verbose=False, backend='scipy')
    t_cpu = time.perf_counter() - t0
    
    # JAX GPU
    t0 = time.perf_counter()
    out_jax, steps_jax = wg.propagate_NEE(pump, v_ref=v_ref, verbose=False, backend='jax')
    t_jax = time.perf_counter() - t0
    
    # Field correlation
    corr = float(np.abs(np.sum(out_cpu.a * np.conj(out_jax.a))) /
                 np.sqrt(np.sum(np.abs(out_cpu.a)**2) * np.sum(np.abs(out_jax.a)**2)))
    
    # --- Row 1: Time domain and PSD using pulse convenience methods ---
    fig, axes = plt.subplots(2, 2, figsize=(15, 10), tight_layout=True)
    fig.suptitle(f'{title}\nCorrelation = {corr:.6f} | '
                 f'CPU: {t_cpu:.2f}s ({len(steps_cpu)} steps) | '
                 f'JAX: {t_jax:.2f}s ({len(steps_jax)} steps)',
                 fontsize=13)
    
    # Time domain overlay (pulse.plot_magsq style)
    ax = axes[0, 0]
    pump.plot_magsq(ax=ax, t_unit='ps')
    out_cpu.plot_magsq(ax=ax, t_unit='ps')
    out_jax.plot_magsq(ax=ax, t_unit='ps')
    ax.get_lines()[-3].set(color='k', linestyle='--', alpha=0.3, label='Input')
    ax.get_lines()[-2].set(color='b', linewidth=1.5, label='CPU (SciPy)')
    ax.get_lines()[-1].set(color='r', linestyle='--', linewidth=1.5, label='JAX GPU')
    ax.set_xlim(-0.5, 0.5)
    ax.legend(); ax.set_title('Temporal Intensity')
    
    # PSD overlay (pulse.plot_PSD style)
    ax = axes[0, 1]
    pump.plot_PSD(ax=ax, f_unit='um')
    out_cpu.plot_PSD(ax=ax, f_unit='um')
    out_jax.plot_PSD(ax=ax, f_unit='um')
    ax.get_lines()[-3].set(color='k', linestyle='--', alpha=0.3, label='Input')
    ax.get_lines()[-2].set(color='b', linewidth=1.5, label='CPU (SciPy)')
    ax.get_lines()[-1].set(color='r', linestyle='--', linewidth=1.5, label='JAX GPU')
    ax.legend(); ax.set_title('Power Spectral Density')
    
    # --- Row 2: Differences ---
    # Time domain difference
    ax = axes[1, 0]
    I_cpu = np.abs(out_cpu.a)**2
    I_jax = np.abs(out_jax.a)**2
    I_peak = max(np.max(I_cpu), np.max(I_jax))
    ax.plot(out_cpu.t/ps, (I_jax - I_cpu) / I_peak * 100)
    ax.set_xlim(-0.5, 0.5)
    ax.set_xlabel('Time (ps)'); ax.set_ylabel('Difference (% of peak)')
    ax.set_title('Temporal Difference (JAX - CPU)'); ax.grid(True)
    
    # Spectral difference
    ax = axes[1, 1]
    spec_cpu = np.abs(fft(out_cpu.a))**2
    spec_jax = np.abs(fft(out_jax.a))**2
    spec_peak = max(np.max(spec_cpu), np.max(spec_jax))
    wl_plot = fftshift(c/f_abs)/um
    ax.plot(wl_plot, fftshift((spec_jax - spec_cpu) / spec_peak * 100))
    ax.set_xlabel('Wavelength (um)'); ax.set_ylabel('Difference (% of peak)')
    ax.set_title('Spectral Difference (JAX - CPU)'); ax.grid(True)
    
    plt.show()
    
    # Summary
    E_in = pump.energy_td()
    print(f'  Energy: in={E_in/pJ:.4f}  cpu={out_cpu.energy_td()/pJ:.4f}  '
          f'jax={out_jax.energy_td()/pJ:.4f} pJ')
    print(f'  Speedup: {t_cpu/t_jax:.1f}x')
    print()
    return corr

## Level 1: Linear propagation (no nonlinearity)

With X0=0, the pulse propagates linearly through the dispersive waveguide.
Both solvers should give identical results.

In [ ]:
pump = pulses.sech_pulse(t, 100*fs, Pavg=1*uW, f_ref=f_ref, f0=c/(2*um),
                         Npwr_dB=200, frep=250*MHz)
wg = make_waveguide(L=4*mm, X0=0)
run_and_compare('Level 1: Linear, lossless, 4mm', pump, wg);

## Level 2: Linear with loss

Add 0.5 dB/cm propagation loss. Output energy should be attenuated by exp(-alpha*L).

In [ ]:
from snow import util
alpha = util.absorption_coeff(0.5)
wg = make_waveguide(L=4*mm, X0=0, alpha=alpha)
run_and_compare('Level 2: Linear, 0.5 dB/cm loss, 4mm', pump, wg);

## Level 3: Weak SHG (uniform QPM)

Low-power SHG in a 4mm TFLN waveguide with 5.18 um poling period.
At 1 uW, we're deep in the undepleted-pump regime.

In [ ]:
wg = make_waveguide(L=4*mm, X0=1.1e-12)
run_and_compare('Level 3: SHG, uniform QPM, 4mm, 1 uW', pump, wg);

## Level 4: Chirped QPM

The poling period varies linearly with z, broadening the phase-matching bandwidth.
This tests that the z-dependent poling lookup table works correctly.

In [ ]:
chirp_rate = 0.5e-6  # period increases by 0.5 um per mm
poling_chirped = lambda z: np.sign(np.cos(z*2*pi/(pp + chirp_rate*z)))
wg = make_waveguide(L=4*mm, X0=1.1e-12, poling_fn=poling_chirped)
run_and_compare('Level 4: Chirped QPM, 4mm', pump, wg);

## Level 5: Apodized QPM

Gaussian-envelope poling suppresses the spectral sidelobes of the SHG.
Tests continuously-varying coupling strength.

In [ ]:
L5 = 4*mm
poling_apod = lambda z: np.exp(-((z - L5/2)/(L5/4))**2) * np.sign(np.cos(z*2*pi/pp))
wg = make_waveguide(L=L5, X0=1.1e-12, poling_fn=poling_apod)
run_and_compare('Level 5: Apodized QPM, 4mm', pump, wg);

## Level 6: Longer crystal (10mm)

More propagation distance means more accumulated error between the solvers.
Tests stability over longer integration.

In [ ]:
wg = make_waveguide(L=10*mm, X0=1.1e-12)
run_and_compare('Level 6: Uniform QPM, 10mm', pump, wg);

## Level 7: SHG with loss

Nonlinear propagation combined with propagation loss.

In [ ]:
alpha = util.absorption_coeff(0.3)
wg = make_waveguide(L=4*mm, X0=1.1e-12, alpha=alpha)
run_and_compare('Level 7: SHG + 0.3 dB/cm loss, 4mm', pump, wg);

## Level 8: Higher power (approaching pump depletion)

At higher powers, nonlinear conversion is stronger and the problem becomes
more sensitive to the exact step sequence. We expect the correlation to drop
but the physics (energy conservation, spectral features) should still agree.

In [ ]:
pump_hi = pulses.sech_pulse(t, 100*fs, Pavg=10*uW, f_ref=f_ref, f0=c/(2*um),
                            Npwr_dB=200, frep=250*MHz)
wg = make_waveguide(L=4*mm, X0=1.1e-12)
run_and_compare('Level 8: SHG, 10 uW, 4mm', pump_hi, wg);

## Level 8b: Power sweep — where does SHG become efficient?

Sweep pump power from 0.1 uW to 1 mW at fixed L=4mm to find the
onset of significant SHG conversion.  In the undepleted regime,
SH power scales as P$_{\mathrm{pump}}^2$.  At higher powers, pump
depletion causes the efficiency to saturate and eventually back-convert.

In [ ]:
import jax.numpy as jnp
poling_jax = lambda z: jnp.sign(jnp.cos(z * 2*jnp.pi / pp))

# Power sweep at fixed L=4mm
L_fix = 4*mm
P_values = np.logspace(-1, 3, 20) * uW  # 0.1 uW to 1 mW

eff_pwr_cpu = []; eff_pwr_jax = []

for Pavg in P_values:
    p = pulses.sech_pulse(t, 100*fs, Pavg=Pavg, f_ref=f_ref, f0=c/(2*um),
                          Npwr_dB=200, frep=250*MHz)
    E_in_p = p.energy_td()
    wg_p = make_waveguide(L=L_fix, X0=1.1e-12)

    out_c, _ = wg_p.propagate_NEE(p, v_ref=v_ref, verbose=False, backend='scipy')
    out_j, _ = wg_p.propagate_NEE(p, v_ref=v_ref, verbose=False,
                                  backend='jax', poling_fn_jax=poling_jax)

    sh_c = out_c.apply_filter(c/(1*um), 50*THz)
    sh_j = out_j.apply_filter(c/(1*um), 50*THz)
    eff_pwr_cpu.append(sh_c.energy_td() / E_in_p)
    eff_pwr_jax.append(sh_j.energy_td() / E_in_p)
    print(f'  P={Pavg/uW:8.1f} uW: cpu={eff_pwr_cpu[-1]:.4e}  jax={eff_pwr_jax[-1]:.4e}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), tight_layout=True)

# Efficiency vs power
ax1.loglog(np.array(P_values)/uW, eff_pwr_cpu, 'bo-', label='CPU', markersize=5)
ax1.loglog(np.array(P_values)/uW, eff_pwr_jax, 'rs--', label='JAX', markersize=5)
# Quadratic reference line
P_ref = np.array([P_values[0], P_values[5]])
eff_ref = eff_pwr_cpu[0] * (P_ref / P_values[0])
ax1.loglog(P_ref/uW, eff_ref, 'k:', alpha=0.5, label='$\\propto P$ (undepleted)')
ax1.set_xlabel('Average power (uW)'); ax1.set_ylabel('SH conversion efficiency')
ax1.set_title(f'SHG Efficiency vs Pump Power (L={L_fix/mm:.0f}mm)')
ax1.legend(); ax1.grid(True, which='both', alpha=0.3)

# SH power vs pump power
sh_pwr_cpu = np.array(eff_pwr_cpu) * np.array(P_values)
sh_pwr_jax = np.array(eff_pwr_jax) * np.array(P_values)
ax2.loglog(np.array(P_values)/uW, sh_pwr_cpu/uW, 'bo-', label='CPU', markersize=5)
ax2.loglog(np.array(P_values)/uW, sh_pwr_jax/uW, 'rs--', label='JAX', markersize=5)
# P^2 reference
P_ref2 = np.array(P_values[:8])
sh_ref2 = sh_pwr_cpu[0] * (P_ref2/P_values[0])**2
ax2.loglog(P_ref2/uW, sh_ref2/uW, 'k:', alpha=0.5, label='$\\propto P^2$ (undepleted)')
ax2.set_xlabel('Pump power (uW)'); ax2.set_ylabel('SH power (uW)')
ax2.set_title('SH Output Power vs Pump Power')
ax2.legend(); ax2.grid(True, which='both', alpha=0.3)

plt.show()

## Level 8c: Power vs length — 2D efficiency map

Sweep both pump power and crystal length to map out the SHG design
space.  This shows the tradeoff between power and interaction length,
including the onset of pump depletion, back-conversion, and the effect
of group velocity mismatch (walkoff) at longer crystals.

This is a 2D parameter sweep — exactly the kind of design exploration
where GPU acceleration pays off.

In [ ]:
# 2D sweep: power x length
P_2d = np.logspace(0, 2.5, 12) * uW     # 1 uW to ~300 uW
L_2d = np.arange(1, 13) * mm             # 1 to 12 mm

eff_map = np.zeros((len(P_2d), len(L_2d)))

print(f'Running {len(P_2d)}x{len(L_2d)} = {len(P_2d)*len(L_2d)} simulations (JAX)...')
t_start = time.perf_counter()

for ip, Pavg in enumerate(P_2d):
    for il, L_val in enumerate(L_2d):
        p = pulses.sech_pulse(t, 100*fs, Pavg=Pavg, f_ref=f_ref, f0=c/(2*um),
                              Npwr_dB=200, frep=250*MHz)
        wg_2d = make_waveguide(L=L_val, X0=1.1e-12)
        out, _ = wg_2d.propagate_NEE(p, v_ref=v_ref, verbose=False,
                                     backend='jax', poling_fn_jax=poling_jax)
        sh = out.apply_filter(c/(1*um), 50*THz)
        eff_map[ip, il] = sh.energy_td() / p.energy_td()
    print(f'  P={Pavg/uW:6.1f} uW done ({ip+1}/{len(P_2d)})')

t_total = time.perf_counter() - t_start
print(f'Total: {t_total:.1f}s ({t_total/len(P_2d)/len(L_2d):.2f}s per point)')

# Plot
fig, ax = plt.subplots(figsize=(10, 6), tight_layout=True)
L_mm_2d = np.array(L_2d)/mm
P_uW_2d = np.array(P_2d)/uW
X, Y = np.meshgrid(L_mm_2d, P_uW_2d)

levels = np.linspace(0, min(0.5, np.max(eff_map)), 20)
cs = ax.contourf(X, Y, eff_map, levels=levels, cmap='viridis')
ax.contour(X, Y, eff_map, levels=[0.01, 0.05, 0.1, 0.2, 0.3],
           colors='white', linewidths=0.8)
cb = plt.colorbar(cs, ax=ax)
cb.set_label('SH Conversion Efficiency')
ax.set_xlabel('Crystal Length (mm)')
ax.set_ylabel('Pump Power (uW)')
ax.set_yscale('log')
ax.set_title('SHG Efficiency: Power vs Crystal Length\n'
             '(100fs sech, 5.18um QPM, TFLN waveguide)')
plt.show()

## Level 9: Parameter sweep — SHG efficiency vs crystal length

A standard design task: how does conversion efficiency depend on crystal
length?  In the undepleted-pump regime, SHG power scales as L$^2$.  At
higher conversion, back-conversion and walk-off cause the efficiency to
roll over.

We use the same N=2^10 grid as the earlier levels.  The JAX-native poling
function (`jnp.sign(jnp.cos(...))`) gives bit-exact evaluation matching
the CPU, so the only divergence source is GPU vs CPU floating-point
arithmetic in the adaptive step controller.

In [ ]:
from IPython.display import display
import jax.numpy as jnp

# JAX-native poling function — bit-exact with the numpy version
poling_jax = lambda z: jnp.sign(jnp.cos(z * 2*jnp.pi / pp))

# Sweep parameters
L_values = np.arange(1, 16) * mm
pump_sweep = pulses.sech_pulse(t, 100*fs, Pavg=1*uW, f_ref=f_ref, f0=c/(2*um),
                               Npwr_dB=200, frep=250*MHz)
E_in = pump_sweep.energy_td()

# Storage
eff_cpu = []; eff_jax = []
t_cpu_list = []; t_jax_list = []

# Create figure (suppress auto-display with ioff)
plt.ioff()
fig_sweep, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), tight_layout=True)
line_cpu, = ax1.plot([], [], 'bo-', label='CPU (SciPy)', markersize=6)
line_jax, = ax1.plot([], [], 'rs--', label='JAX GPU', markersize=6)
ax1.set_xlabel('Crystal length (mm)'); ax1.set_ylabel('SH conversion efficiency')
ax1.set_title('SHG Efficiency vs Crystal Length')
ax1.legend(); ax1.grid(True); ax1.set_xlim(0, 16)
ax2.set_xlabel('Crystal length (mm)'); ax2.set_ylabel('Wall time (s)')
ax2.set_title('Computation Time per Point'); ax2.grid(True, axis='y')
fig_handle = display(fig_sweep, display_id=True)
plt.ion()

for i, L_val in enumerate(L_values):
    wg_s = make_waveguide(L=L_val, X0=1.1e-12)

    # CPU
    t0 = time.perf_counter()
    out_c, _ = wg_s.propagate_NEE(pump_sweep, v_ref=v_ref, verbose=False, backend='scipy')
    dt_cpu = time.perf_counter() - t0

    # JAX with native poling function
    t0 = time.perf_counter()
    out_j, _ = wg_s.propagate_NEE(pump_sweep, v_ref=v_ref, verbose=False,
                                  backend='jax', poling_fn_jax=poling_jax)
    dt_jax = time.perf_counter() - t0

    # SH efficiency: filter around 1 um
    sh_cpu = out_c.apply_filter(c/(1*um), 50*THz)
    sh_jax = out_j.apply_filter(c/(1*um), 50*THz)
    eff_cpu.append(sh_cpu.energy_td() / E_in)
    eff_jax.append(sh_jax.energy_td() / E_in)
    t_cpu_list.append(dt_cpu)
    t_jax_list.append(dt_jax)

    # Update plots
    L_mm = [l/mm for l in L_values[:i+1]]
    line_cpu.set_data(L_mm, eff_cpu)
    line_jax.set_data(L_mm, eff_jax)
    ax1.relim(); ax1.autoscale_view()

    ax2.clear()
    x_bar = np.array(L_mm); w = 0.3
    ax2.bar(x_bar - w/2, t_cpu_list, w, label='CPU', color='steelblue')
    ax2.bar(x_bar + w/2, t_jax_list, w, label='JAX', color='indianred')
    ax2.set_xlabel('Crystal length (mm)'); ax2.set_ylabel('Wall time (s)')
    ax2.set_title('Computation Time per Point')
    ax2.legend(); ax2.grid(True, axis='y')

    fig_handle.update(fig_sweep)
    print(f'  L={L_val/mm:2.0f}mm: CPU={dt_cpu:.1f}s  JAX={dt_jax:.1f}s  '
          f'eff_cpu={eff_cpu[-1]:.4e}  eff_jax={eff_jax[-1]:.4e}')

total_cpu = sum(t_cpu_list); total_jax = sum(t_jax_list)
print(f'\n  Total: CPU={total_cpu:.1f}s  JAX={total_jax:.1f}s  '
      f'Speedup={total_cpu/total_jax:.1f}x over {len(L_values)} points')

## Level 10: Broadband simulation (N=2^12)

Levels 1-8 used N=2^10 (1024 points), where the CPU is fast and JAX pays
compilation overhead.  Real broadband simulations -- spanning multiple octaves
for supercontinuum or OPA -- need N=2^12 to 2^16.

Here we use N=2^12 with a 10mm crystal.  At this grid size, the CPU FFTs
become the bottleneck and JAX wins convincingly.

In [ ]:
# Broadband grid: 500nm to 5um, N=2^12
N10 = 2**12
f_max10, f_min10 = c/(500*nm), c/(5*um)
BW10 = f_max10 - f_min10
dt10 = 1/BW10
t10 = -N10/(2*BW10) + np.arange(0, N10/BW10, step=dt10)
f_ref10 = (f_max10 + f_min10)/2
f_abs10 = fftfreq(N10, dt10) + f_ref10

pump10 = pulses.sech_pulse(t10, 100*fs, Pavg=1*uW, f_ref=f_ref10, f0=c/(2*um),
                           Npwr_dB=200, frep=250*MHz)
wg10 = make_waveguide(L=10*mm, X0=1.1e-12)
v_ref10 = 1/wg10.beta1(1*um)

# Use broadband grid for plotting
_f_abs_save = f_abs
f_abs = f_abs10
run_and_compare('Level 10: Broadband N=2^12, 10mm', pump10, wg10, v_ref=v_ref10)
f_abs = _f_abs_save

## Summary

The JAX GPU solver is a faithful port of SciPy's RK45 (Dormand-Prince) step
controller, including FSAL and identical safety/factor constants.  When called
directly with the same inputs, the two solvers produce **bit-identical** results
(same step counts, same efficiencies to 8+ digits).

Through the waveguide API with `poling_fn_jax`, the solvers match perfectly
through ~10mm.  At longer crystals, GPU vs CPU floating-point arithmetic can
cause a single accept/reject decision to differ, after which the adaptive
trajectories diverge.  Both remain physically valid (energy conserving,
correct spectral features).

**Performance**: CPU time scales linearly with crystal length; JAX time is
nearly constant (~5-6s at N=1024) due to `lax.while_loop` overhead.
Crossover occurs around L=8mm.  At larger N (2^12+), JAX wins at any length.